In [17]:
import pandas as pd
import os
# os.listdir('/content/drive/MyDrive/global-seismic-trends/')

df = pd.read_csv('/content/drive/MyDrive/global-seismic-trends/data/earthquakes.csv')
df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,net,code,ids,sources,types,nst,dmin,rms,gap,type
0,av91018173,1612051063470,1612139465880,51.841833,-178.581000,22.48,1.44,ml,"134 km W of Adak, Alaska",reviewed,...,av,91018173,",av91018173,",",av,",",origin,phase-data,",10.0,NaN,0.15,174.0,earthquake
1,nc73518690,1612050679680,1612472529323,38.030167,-118.800167,9.59,1.87,md,"27km SE of Bodie, CA",reviewed,...,nc,73518690,",nc73518690,",",nc,",",nearby-cities,origin,phase-data,scitech-link,",25.0,0.2920,0.05,103.0,earthquake
2,pr2021030122,1612049970820,1671209101825,17.951600,-66.985300,9.00,1.65,md,"6 km ESE of La Parguera, Puerto Rico",reviewed,...,pr,2021030122,",pr2021030122,",",pr,",",origin,phase-data,",4.0,0.0616,0.05,267.0,earthquake
3,ak0211e015ln,1612049787503,1613163018719,59.183200,-136.452400,7.60,1.70,ml,"35 km SW of Covenant Life, Alaska",reviewed,...,ak,0211e015ln,",ak0211e015ln,",",ak,",",origin,phase-data,",NaN,NaN,0.59,NaN,earthquake
4,ak0211dzz7sr,1612049243670,1613163018483,61.024800,-150.391900,16.60,1.40,ml,"19 km NE of Point Possession, Alaska",reviewed,...,ak,0211dzz7sr,",ak0211dzz7sr,",",ak,",",dyfi,origin,phase-data,",NaN,NaN,0.65,NaN,earthquake


In [18]:
df.shape

(849573, 26)

In [19]:
df.columns
# df.columns.tolist()

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'alert', 'felt', 'cdi', 'mmi',
       'sig', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms',
       'gap', 'type'],
      dtype='object')

In [20]:
# df.describe
# df.info

In [21]:
# df.isna().sum() # Checking all the null values column wise

In [22]:
df.isnull().sum().sort_values(ascending=False)

,0
alert,844736
mmi,838721
felt,807459
cdi,807459
dmin,234841
nst,201309
gap,173944
rms,76
mag,75
magType,75


In [23]:
# Working DF
df_working = df.copy()
# df_working.tail()

df_working['id'].duplicated().sum()

df_working['time'] = pd.to_datetime(df['time'], unit='ms')
df_working['year'] = df_working['time'].dt.year
df_working['month'] = df_working['time'].dt.month
df_working['day'] = df_working['time'].dt.day
df_working['hour'] = df_working['time'].dt.hour
df_working['weekday'] = df_working['time'].dt.day_name()
df_working[ ['year', 'month', 'day', 'hour', 'weekday'] ].sort_values(by='year', ascending=True)

# df_working[ ['year', 'month', 'day', 'hour', 'weekday'] ].isnull().sum()

# df_working[df_working['hour'] == 0][["time", "hour"]].head()

df_working['region'] = df['place'].str.split(',').str[-1].str.strip()
df_working['region'].value_counts().head()



,count
region,
Alaska,279595
CA,227333
Hawaii,45709
Nevada,27642
Texas,26906


In [ ]:
# setting magnitute category

def setup_magnitude_category(mag):
  if mag<2:
    return 'Minor'
  elif mag<4:
    return 'Light'
  elif mag<6:
    return 'Moderate'
  elif mag<7:
    return 'Strong'
  else:
    return 'Major'

df_working['magnitude_category'] = df_working['mag'].apply(setup_magnitude_category)
df_working['magnitude_category'].value_counts()

,count
magnitude_category,
Minor,604690
Light,152842
Moderate,91194
Strong,685
Major,162


In [ ]:
# set depth category
def set_depth_category(depth):
  if depth<70:
    return 'Shallow'
  elif depth<300:
    return 'Intermediate'
  else:
    return 'Deep'

# df_working.columns
df_working['depth_category'] = df_working['depth_km'].apply(set_depth_category)
df_working['depth_category'].value_counts()

,count
depth_category,
Shallow,769282
Intermediate,73484
Deep,6807


In [ ]:
# df_working.info()


In [ ]:
#drop null rows of mag - as we dont know the accurate value
df_working = df_working[df_working['mag'].notna()]

df_working['rms'] = df_working['rms'].fillna(df_working['rms'].median())
df_working['alert'] = df_working['alert'].fillna('none')
df_working['mmi'] = df_working['mmi'].fillna(0)
df_working['cdi'] = df_working['cdi'].fillna(0)
df_working['felt'] = df_working['felt'].fillna(0)
df_working['nst'] = df_working['nst'].fillna(0)
df_working['gap'] = df_working['gap'].fillna(-1)
df_working['dmin'] = df_working['dmin'].fillna(df_working['dmin'].median())

df_working.isnull().sum().sort_values(ascending=False)

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
magType,0
place,0
status,0


In [ ]:
# save csv for future SQL & reports
df_working.to_csv('/content/drive/MyDrive/global-seismic-trends/data/earthquakes_clean.csv', index=False)

In [ ]:
# Fetching casualities data for the report
# df[df['types'].str.contains('losspager')] #4837 rows available



In [ ]:
import requests
import pandas as pd

impacts_df = df[df['types'].str.contains('losspager', na=False)].copy()

impact_records = []

total = len(impacts_df)

for index, event_id in enumerate(impacts_df["id"], start=1):

    progress = (index / total) * 100
    print(
        f"\rProgress: {progress:6.2f}% ({index}/{total}) | Event: {event_id}",
        end=""
    )

    try:
        detail_url = (
            "https://earthquake.usgs.gov/fdsnws/event/1/query"
            f"?eventid={event_id}&format=geojson"
        )

        detail = requests.get(detail_url, timeout=30).json()

        products = detail.get("properties", {}).get("products", {})

        if "losspager" not in products:
            continue

        contents = products["losspager"][0].get("contents", {})

        if "json/losses.json" not in contents:
            continue

        loss_url = contents["json/losses.json"]["url"]

        loss_data = requests.get(loss_url, timeout=30).json()

        impact_records.append({
            "event_id": event_id,
            "empirical_fatalities": (
                loss_data.get("empirical_fatality", {})
                         .get("total_fatalities", 0)
            ),
            "semi_empirical_fatalities": (
                loss_data.get("semi_empirical_fatalities", {})
                         .get("fatalities", 0)
            ),
            "economic_loss_usd": (
                loss_data.get("empirical_economic", {})
                         .get("total_dollars", 0)
            )
        })

    except Exception:
        continue

print("\n Losses data saved.")

impact_df = pd.DataFrame(impact_records)
print(f"Total Impact Records Collected: {len(impact_df)}")


Progress: 100.00% (4837/4837) | Event: us6000tz1c
 Losses data saved.
Total Impact Records Collected: 4827


In [ ]:
# Save CSV
csv_path = (
    "/content/drive/MyDrive/global-seismic-trends/"
    "data/earthquake_impacts_enriched.csv"
)

impact_df.to_csv(csv_path, index=False)

print(f"CSV saved: {csv_path}")

impact_df.head()

CSV saved: /content/drive/MyDrive/global-seismic-trends/data/earthquake_impacts_enriched.csv


,event_id,empirical_fatalities,semi_empirical_fatalities,economic_loss_usd
0,us6000dd7l,0,0,0
1,us6000dd2i,0,0,0
2,us6000dcxt,0,0,0
3,us6000dcx7,0,0,5135
4,us6000dcx1,0,0,80468


In [ ]:
impact_df.isnull().sum()

,0
event_id,0
empirical_fatalities,0
semi_empirical_fatalities,0
economic_loss_usd,0


In [24]:
unique_locations = df_working[
    ["latitude", "longitude"]
].drop_duplicates()

len(unique_locations)

824107

In [25]:
# !pip install reverse_geocoder # to get country & continent based on lag, log

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 20.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268112 sha256=7eb770e0c5c10b36d2dc586ecdc8a069737b84d245860b5c85e06f20656b276e
  Stored in directory: /root/.cache/pip/wheels/61/62/ae/e0593fd403b7274fafb7a534ed54f9f880d0b5840963d64704
Successfully built reverse_geocoder


In [26]:
import reverse_geocoder as rg

coords = list(
    zip(
        unique_locations["latitude"],
        unique_locations["longitude"]
    )
)

results = rg.search(coords)

Loading formatted geocoded file...


In [27]:
unique_locations["country_code"] = [
    r["cc"] for r in results
]

unique_locations["country"] = [
    r["name"] for r in results
]

In [29]:
# !pip install pycountry-convert

In [30]:
import pycountry_convert as pc

def country_to_continent(country_code):
    try:
        continent_code = pc.country_alpha2_to_continent_code(
            country_code
        )
        return {
            "AF": "Africa",
            "AS": "Asia",
            "EU": "Europe",
            "NA": "North America",
            "OC": "Oceania",
            "SA": "South America",
            "AN": "Antarctica"
        }[continent_code]
    except:
        return "Unknown"

unique_locations["continent"] = (
    unique_locations["country_code"]
    .apply(country_to_continent)
)

In [31]:
df_working = df_working.merge(
    unique_locations,
    on=["latitude", "longitude"],
    how="left"
)

In [33]:
# df_working[ ["country", "country_code", "continent"]].head() # test row values

In [35]:
# save df as CSV for SQL report with country and continent
df_working.to_csv('/content/drive/MyDrive/global-seismic-trends/data/earthquakes_clean.csv', index=False)